# Chalkboards and Dashboards: AI-Augmented Analysis of Massachusetts Schools
### Using the OpenAI API to generate deeper insights from the MA Department of Education dataset

---

**Project Goal:** Augment traditional Tableau data analysis with LLM-generated insights  
**Dataset:** Massachusetts Public Schools 2017 -- 1,861 schools, 299 columns  
**Source:** Kaggle / Massachusetts Department of Education (profiles.doe.mass.edu)  
**API Used:** OpenAI `gpt-4o-mini`  

---

## What This Notebook Does
1. Loads and prepares the MA Schools dataset
2. Classifies high schools using graduation, college attendance, SAT and AP metrics
3. Classifies K-8 schools using MCAS 3rd and 4th grade scores
4. Classifies middle schools using MCAS 6th, 7th and 8th grade scores
5. Combines all three classifications into one enriched dataset
6. Generates narrative summaries for the top and bottom performing high schools
7. Validates the AI output -- tier progression, spot checks, consistency
8. Compares AI classification to rule-based Excel IF classification
9. Compares AI classification to the official Massachusetts state accountability levels
10. Exports enriched CSVs for Tableau

---

## Why Three Separate Classifications?

Different school types track different metrics:
- **High schools** track graduation rates, college attendance, SAT and AP scores
- **K-8 schools** track MCAS performance in grades 3 and 4
- **Middle schools** track MCAS performance in grades 6, 7 and 8

Using a single system for all schools would be meaningless -- you cannot ask an elementary school for its graduation rate. Each school type needs its own framework based on the metrics that actually apply to it.

---

## Why Build Our Own Classification When the State Already Has One?

The Massachusetts DOE already classifies schools using an official Level 1-5 accountability system. We built our own for three reasons:

1. **Different question** -- The state measures equity (are achievement gaps closing?). We measure absolute outcomes (are students graduating and attending college?)
2. **Communication** -- "At Risk" is immediately understood. "Level 3 -- Among lowest performing 20% of subgroups" is bureaucratic
3. **Demonstration** -- Comparing our AI classification to the state's official classification produces a genuinely interesting finding that neither system reveals alone

> **Spoiler:** The comparison reveals 43 schools that look like star performers on graduation and college attendance but are simultaneously failing on equity metrics

---
## Step 1: Install and Import Libraries

In [1]:
# Install required libraries (run once)
# !pip install openai pandas openpyxl

In [1]:
from openai import OpenAI
import pandas as pd
import json
import time
import os

print('Libraries loaded successfully!')

Libraries loaded successfully!


---
## Step 2: Set Up the OpenAI Client

Get your API key at: https://platform.openai.com/api-keys

> **Important:** Never share your API key or push it to GitHub.  
> Replace the key with `your-openai-key-here` before uploading this file anywhere.

In [ ]:
os.environ['OPENAI_API_KEY'] = 'your-key-here'
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print(f'Client initialized. Using model: {MODEL}')

Client initialized. Using model: gpt-4o-mini


---
## Step 3: Load the Dataset

In [3]:
df = pd.read_excel('MA_Schools_data_set.xlsx')
print(f'Total schools: {len(df)}')
print(f'Columns: {len(df.columns)}')
df.head(3)

Total schools: 1861
Columns: 299


,School Code,School Name,School Name Short,School Type,Address 1,Town,State,Zip,Latitude,Longitude,...,MCAS_10thGrade_English_Incl. in SGP(#),Accountability and Assistance Level,Accountability and Assistance Description,School Accountability Percentile (1-99),Progress and Performance Index (PPI) - All Students,Progress and Performance Index (PPI) - High Needs Students,District_Accountability and Assistance Level,District_Accountability and Assistance Description,District_Progress and Performance Index (PPI) - All Students,District_Progress and Performance Index (PPI) - High Needs Students
0,10505,Abington High,Abington High,Public School,201 Gliniewicz Way,Abington,MA,2351,42.117505,-70.955010,...,111.0,Level 1,Meeting gap narrowing goals,42.0,76.0,75.0,Level 3,One or more schools in the district classified...,63.0,60.0
1,10003,Beaver Brook Elementary School,Beaver Brook El,Public School,1 Ralph Hamlin Lane,Abington,MA,2351,42.118543,-70.946355,...,NaN,Level 3,Among lowest performing 20% of subgroups,34.0,69.0,73.0,Level 3,One or more schools in the district classified...,63.0,60.0
2,10002,Center Elementary School,Ctr Elem,Public School,201 Gliniewicz Way,Abington,MA,2351,42.117505,-70.955010,...,NaN,Insufficient data,NaN,NaN,NaN,NaN,Level 3,One or more schools in the district classified...,63.0,60.0


---
## Step 4: Define Classification Prompts

We use the same 4 tiers for all school types to keep the Tableau dashboard consistent.  
The prompt changes the context and metrics based on school type but the output format is identical.

**Why JSON output?**  
Asking the model to respond in JSON means we can automatically parse the response into dataframe columns without any manual text processing.

In [4]:
JSON_FORMAT = """
Respond ONLY with valid JSON in this exact format, no other text:
{{
  "classification": "<tier>",
  "insight": "<one sentence max 20 words>",
  "key_strength": "<one word or short phrase>",
  "key_concern": "<one word or short phrase>"
}}
"""

TIERS = """
Classify the school into ONE of these performance tiers:
   - "Thriving"   -- strong outcomes across most metrics
   - "Performing" -- average outcomes, no major red flags
   - "Struggling" -- below average outcomes in multiple key areas
   - "At Risk"    -- significantly below average, needs urgent intervention
"""

HS_PROMPT = """
You are an education data analyst reviewing Massachusetts HIGH SCHOOL data from 2017.
Key metrics for high schools are graduation rate, college attendance, dropout rate, SAT and AP scores.
""" + TIERS + """
Also write a single sentence insight capturing the most notable thing about this school.
School Data:
{school_data}
""" + JSON_FORMAT

K8_PROMPT = """
You are an education data analyst reviewing Massachusetts K-8 ELEMENTARY SCHOOL data from 2017.
Key metrics are MCAS proficiency rates in 3rd and 4th grade math and english, warning/fail rates,
and student demographic needs.
""" + TIERS + """
Also write a single sentence insight capturing the most notable thing about this school.
School Data:
{school_data}
""" + JSON_FORMAT

MIDDLE_PROMPT = """
You are an education data analyst reviewing Massachusetts MIDDLE SCHOOL data from 2017.
Key metrics are MCAS proficiency rates in 6th, 7th and 8th grade math and english, warning/fail rates,
and student demographic needs.
""" + TIERS + """
Also write a single sentence insight capturing the most notable thing about this school.
School Data:
{school_data}
""" + JSON_FORMAT

print('Prompt templates defined!')

Prompt templates defined!


---
## Step 5: Define Helper Functions

One reusable `classify_school()` function handles all three school types.  
One reusable `run_classification()` function handles the loop for any school group.

In [5]:
def classify_school(row, prompt_template):
    """
    Sends one school's data to the OpenAI API and returns a classification.
    Args:
        row: one row from a pandas dataframe
        prompt_template: HS_PROMPT, K8_PROMPT or MIDDLE_PROMPT
    Returns:
        dict with classification, insight, key_strength, key_concern
    """
    school_data_str = '\n'.join([
        f'  {k}: {v}'
        for k, v in row.to_dict().items()
        if pd.notna(v)
    ])
    prompt = prompt_template.format(school_data=school_data_str)
    try:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=200,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return json.loads(response.choices[0].message.content.strip())
    except Exception as e:
        return {'classification': 'Error', 'insight': str(e),
                'key_strength': 'N/A', 'key_concern': 'N/A'}


def run_classification(df_input, prompt_template, label):
    """Runs classification on all schools in a dataframe with progress updates."""
    classifications, insights, strengths, concerns = [], [], [], []
    total = len(df_input)
    print(f'Classifying {total} {label} schools...')
    for i, (_, row) in enumerate(df_input.iterrows()):
        result = classify_school(row, prompt_template)
        classifications.append(result.get('classification', 'Unknown'))
        insights.append(result.get('insight', ''))
        strengths.append(result.get('key_strength', ''))
        concerns.append(result.get('key_concern', ''))
        if (i + 1) % 10 == 0:
            print(f'  Processed {i+1}/{total}...')
        time.sleep(0.3)
    df_input = df_input.copy()
    df_input['AI_Classification'] = classifications
    df_input['AI_Insight'] = insights
    df_input['AI_Key_Strength'] = strengths
    df_input['AI_Key_Concern'] = concerns
    df_input['School_Level'] = label
    print(f'\nDone! Classification breakdown:')
    print(df_input['AI_Classification'].value_counts())
    return df_input


print('Helper functions defined!')

Helper functions defined!


---
# PART 1: High School Classification

**Filter logic:** Schools with both `% Graduated` and `% Attending College` data are high schools.  
These metrics only exist for schools with a graduating class -- no elementary or middle school will have them.  
Filtering on graduation data captures all schools with a graduating class -- including K-12, 6-12 and alternative high schools that would 
be missed by filtering on Grade = '09,10,11,12' alone. The Grade column is clean but too narrow -- it only captures 269 pure high schools versus 343 schools that actually have graduating classes



In [6]:
hs = df[
    df['% Graduated'].notna() &
    df['% Attending College'].notna()
].copy()

hs_cols = [
    'School Name', 'School Name Short', 'Town', 'School Type',
    'TOTAL_Enrollment', 'Average Class Size',
    '% Economically Disadvantaged', '% High Needs',
    '% English Language Learner', '% Students With Disabilities',
    '% Graduated', '% Dropped Out', '% Attending College',
    'Average SAT_Math', 'Average SAT_Reading', '% AP_Score 3-5',
    'Average Expenditures per Pupil',
    'Accountability and Assistance Level',
    'School Accountability Percentile (1-99)'
]
hs_cols = [c for c in hs_cols if c in hs.columns]
hs = hs[hs_cols].reset_index(drop=True)
print(f'High schools ready for classification: {len(hs)}')

High schools ready for classification: 343


In [7]:
# Test on one school before running the full batch
test = hs.iloc[0]
print(f"Testing on: {test['School Name']}")
print('-' * 50)
result = classify_school(test, HS_PROMPT)
print(f"Classification : {result['classification']}")
print(f"Insight        : {result['insight']}")
print(f"Key Strength   : {result['key_strength']}")
print(f"Key Concern    : {result['key_concern']}")

Testing on: Abington High
--------------------------------------------------
Classification : Performing
Insight        : Abington High boasts a strong graduation rate and solid college attendance.
Key Strength   : Graduation Rate
Key Concern    : SAT Scores


In [8]:
hs = run_classification(hs, HS_PROMPT, 'High School')

Classifying 343 High School schools...
  Processed 10/343...
  Processed 20/343...
  Processed 30/343...
  Processed 40/343...
  Processed 50/343...
  Processed 60/343...
  Processed 70/343...
  Processed 80/343...
  Processed 90/343...
  Processed 100/343...
  Processed 110/343...
  Processed 120/343...
  Processed 130/343...
  Processed 140/343...
  Processed 150/343...
  Processed 160/343...
  Processed 170/343...
  Processed 180/343...
  Processed 190/343...
  Processed 200/343...
  Processed 210/343...
  Processed 220/343...
  Processed 230/343...
  Processed 240/343...
  Processed 250/343...
  Processed 260/343...
  Processed 270/343...
  Processed 280/343...
  Processed 290/343...
  Processed 300/343...
  Processed 310/343...
  Processed 320/343...
  Processed 330/343...
  Processed 340/343...

Done! Classification breakdown:
AI_Classification
Performing    164
Thriving      115
Struggling     36
At Risk        27
Error           1
Name: count, dtype: int64


---
# PART 2: K-8 School Classification

**Filter logic:** No graduation data (not a high school) AND has 4th grade MCAS scores.  Using 4th grade MCAS as the anchor captures all elementary schools regardless of whether they start at PK, K or grade 1. A Grade-based filter would miss schools with non-standard grade ranges like PK,K,01,02,03,04,05,06 which still serve elementary grades but would not match a simple elementary school label.

In [9]:
k8 = df[
    df['% Graduated'].isna() &
    df['% MCAS_4thGrade_Math_P+A'].notna()
].copy()

k8_cols = [
    'School Name', 'School Name Short', 'Town', 'School Type',
    'TOTAL_Enrollment', 'Average Class Size',
    '% Economically Disadvantaged', '% High Needs',
    '% English Language Learner', '% Students With Disabilities',
    '% MCAS_3rdGrade_Math_P+A', '% MCAS_4thGrade_Math_P+A',
    '% MCAS_3rdGrade_English_P+A', '% MCAS_4thGrade_English_P+A',
    '% MCAS_3rdGrade_Math_W/F', '% MCAS_4thGrade_Math_W/F',
    '% MCAS_3rdGrade_English_W/F', '% MCAS_4thGrade_English_W/F',
    'Average Expenditures per Pupil',
    'Accountability and Assistance Level',
    'School Accountability Percentile (1-99)'
]
k8_cols = [c for c in k8_cols if c in k8.columns]
k8 = k8[k8_cols].reset_index(drop=True)
print(f'K-8 schools ready for classification: {len(k8)}')

K-8 schools ready for classification: 254


In [10]:
test_k8 = k8.iloc[0]
print(f"Testing on: {test_k8['School Name']}")
print('-' * 50)
result = classify_school(test_k8, K8_PROMPT)
print(f"Classification : {result['classification']}")
print(f"Insight        : {result['insight']}")
print(f"Key Strength   : {result['key_strength']}")
print(f"Key Concern    : {result['key_concern']}")

Testing on: Beaver Brook Elementary School
--------------------------------------------------
Classification : Performing
Insight        : Beaver Brook Elementary School has decent proficiency rates but shows room for improvement in 4th-grade math and English.
Key Strength   : Proficiency
Key Concern    : Intervention


In [11]:
k8 = run_classification(k8, K8_PROMPT, 'K-8')

Classifying 254 K-8 schools...
  Processed 10/254...
  Processed 20/254...
  Processed 30/254...
  Processed 40/254...
  Processed 50/254...
  Processed 60/254...
  Processed 70/254...
  Processed 80/254...
  Processed 90/254...
  Processed 100/254...
  Processed 110/254...
  Processed 120/254...
  Processed 130/254...
  Processed 140/254...
  Processed 150/254...
  Processed 160/254...
  Processed 170/254...
  Processed 180/254...
  Processed 190/254...
  Processed 200/254...
  Processed 210/254...
  Processed 220/254...
  Processed 230/254...
  Processed 240/254...
  Processed 250/254...

Done! Classification breakdown:
AI_Classification
Performing    136
Thriving       71
Struggling     41
At Risk         5
Error           1
Name: count, dtype: int64


---
# PART 3: Middle School Classification

**Filter logic:** No graduation data (not high school) AND no 4th grade MCAS (not K-8) AND has 8th grade MCAS scores.  
8th grade is the last year of middle school in Massachusetts and the most universally reported MCAS grade for this school type.

In [12]:
middle = df[
    df['% Graduated'].isna() &
    df['% MCAS_4thGrade_Math_P+A'].isna() &
    df['% MCAS_8thGrade_Math_P+A'].notna()
].copy()

middle_cols = [
    'School Name', 'School Name Short', 'Town', 'School Type',
    'TOTAL_Enrollment', 'Average Class Size',
    '% Economically Disadvantaged', '% High Needs',
    '% English Language Learner', '% Students With Disabilities',
    '% MCAS_6thGrade_Math_P+A', '% MCAS_7thGrade_Math_P+A', '% MCAS_8thGrade_Math_P+A',
    '% MCAS_6thGrade_English_P+A', '% MCAS_7thGrade_English_P+A', '% MCAS_8thGrade_English_P+A',
    '% MCAS_6thGrade_Math_W/F', '% MCAS_7thGrade_Math_W/F', '% MCAS_8thGrade_Math_W/F',
    '% MCAS_6thGrade_English_W/F', '% MCAS_7thGrade_English_W/F', '% MCAS_8thGrade_English_W/F',
    'Average Expenditures per Pupil',
    'Accountability and Assistance Level',
    'School Accountability Percentile (1-99)'
]
middle_cols = [c for c in middle_cols if c in middle.columns]
middle = middle[middle_cols].reset_index(drop=True)
print(f'Middle schools ready for classification: {len(middle)}')

Middle schools ready for classification: 81


In [13]:
test_middle = middle.iloc[0]
print(f"Testing on: {test_middle['School Name']}")
print('-' * 50)
result = classify_school(test_middle, MIDDLE_PROMPT)
print(f"Classification : {result['classification']}")
print(f"Insight        : {result['insight']}")
print(f"Key Strength   : {result['key_strength']}")
print(f"Key Concern    : {result['key_concern']}")

Testing on: Frolio Middle School
--------------------------------------------------
Classification : Performing
Insight        : Frolio MS demonstrates strong performance in English while showing room for improvement in math proficiency.
Key Strength   : English proficiency
Key Concern    : Math outcomes


In [14]:
middle = run_classification(middle, MIDDLE_PROMPT, 'Middle School')

Classifying 81 Middle School schools...
  Processed 10/81...
  Processed 20/81...
  Processed 30/81...
  Processed 40/81...
  Processed 50/81...
  Processed 60/81...
  Processed 70/81...
  Processed 80/81...

Done! Classification breakdown:
AI_Classification
Performing    47
Thriving      18
Struggling    12
At Risk        3
Error          1
Name: count, dtype: int64


---
# PART 4: Combine All Three Classifications

In [15]:
common_cols = [
    'School Name', 'School Name Short', 'Town', 'School Type',
    'TOTAL_Enrollment', 'Average Class Size',
    '% Economically Disadvantaged', '% High Needs',
    'Average Expenditures per Pupil',
    'Accountability and Assistance Level',
    'School Accountability Percentile (1-99)',
    'AI_Classification', 'AI_Insight',
    'AI_Key_Strength', 'AI_Key_Concern', 'School_Level'
]

hs_export     = hs[[c for c in common_cols if c in hs.columns]]
k8_export     = k8[[c for c in common_cols if c in k8.columns]]
middle_export = middle[[c for c in common_cols if c in middle.columns]]

all_schools = pd.concat([hs_export, k8_export, middle_export], ignore_index=True)

print('Combined dataset:')
print(f'  High schools:   {len(hs_export)}')
print(f'  K-8 schools:    {len(k8_export)}')
print(f'  Middle schools: {len(middle_export)}')
print(f'  Total:          {len(all_schools)}')
print()
print('Overall classification breakdown:')
print(all_schools['AI_Classification'].value_counts())
print()
print('Classification by school level:')
print(all_schools.groupby(['School_Level', 'AI_Classification']).size().unstack(fill_value=0))

Combined dataset:
  High schools:   343
  K-8 schools:    254
  Middle schools: 81
  Total:          678

Overall classification breakdown:
AI_Classification
Performing    347
Thriving      204
Struggling     89
At Risk        35
Error           3
Name: count, dtype: int64

Classification by school level:
AI_Classification  At Risk  Error  Performing  Struggling  Thriving
School_Level                                                       
High School             27      1         164          36       115
K-8                      5      1         136          41        71
Middle School            3      1          47          12        18


---
# PART 5: High School Narrative Summaries

AI generated executive summaries for the top and bottom 10 performing high schools.  
Written for a school board audience -- plain language, no jargon.

In [16]:
SUMMARY_PROMPT = """
You are writing an executive summary for the Massachusetts Department of Education.
The audience is school board members who are not data experts.
Below is data on the {group} performing high schools in Massachusetts (2017).
{school_data}
Write a clear, engaging 3-4 sentence paragraph that:
- Summarizes the key patterns you see
- Highlights the most important finding
- Uses plain language (no jargon)
- Mentions 1-2 specific schools by name
Do not use bullet points. Write in flowing prose.
"""

def generate_summary(schools_df, group_label):
    prompt = SUMMARY_PROMPT.format(group=group_label, school_data=schools_df.to_string(index=False))
    response = client.chat.completions.create(
        model=MODEL, max_tokens=400,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response.choices[0].message.content.strip()

summary_cols = ['School Name', 'Town', '% Graduated', '% Attending College',
                '% Economically Disadvantaged', 'Average Class Size']
summary_cols = [c for c in summary_cols if c in hs.columns]

top_10    = hs.nlargest(10, '% Graduated')[summary_cols]
bottom_10 = hs.nsmallest(10, '% Graduated')[summary_cols]

print('Generating summaries...')
top_summary    = generate_summary(top_10, 'TOP 10')
bottom_summary = generate_summary(bottom_10, 'BOTTOM 10')

print('\n' + '='*60)
print('TOP 10 HIGH SCHOOLS - AI SUMMARY')
print('='*60)
print(top_summary)
print('\n' + '='*60)
print('BOTTOM 10 HIGH SCHOOLS - AI SUMMARY')
print('='*60)
print(bottom_summary)

Generating summaries...

TOP 10 HIGH SCHOOLS - AI SUMMARY
The top-performing high schools in Massachusetts consistently demonstrate impressive graduation rates, with all but two schools achieving a remarkable 100% graduation. Notably, Advanced Math and Science Academy Charter School and Bristol County Agricultural High stand out not only for their perfect graduation status but also for their robust college attendance rates, showcasing the effectiveness of their educational programs. Furthermore, while many of these high-achieving schools serve a diverse student body, the percentage of economically disadvantaged students remains relatively low, indicating a potential area for growth in inclusivity. Overall, these schools set a high standard for academic success and college readiness, serving as models for educational excellence across the state.

BOTTOM 10 HIGH SCHOOLS - AI SUMMARY
The data reveals concerning trends among the bottom-performing high schools in Massachusetts, with graduat

---
# PART 6: Validating the AI Output

A good analyst never blindly trusts model output. This section tests whether the AI classifications are accurate, consistent and free of obvious errors.

**This is the most important section** for demonstrating that AI was used as a tool, not a crutch.

### 6a. Tier Progression Check -- Do Classifications Match the Numbers?

In [17]:
validation_cols = [
    'AI_Classification', '% Graduated', '% Attending College',
    '% Dropped Out', '% Economically Disadvantaged',
    'School Accountability Percentile (1-99)'
]
validation_cols = [c for c in validation_cols if c in hs.columns]
tier_summary = hs[validation_cols].groupby('AI_Classification').mean().round(1)

print('Average metrics per AI Classification tier (High Schools):')
print('='*70)
print(tier_summary.to_string())
print()
print('GOOD SIGN: Thriving > Performing > Struggling > At Risk across graduation and college')
print('RED FLAG:  If tiers are mixed up -- the prompt needs refinement')

Average metrics per AI Classification tier (High Schools):
                   % Graduated  % Attending College  % Dropped Out  % Economically Disadvantaged  School Accountability Percentile (1-99)
AI_Classification                                                                                                                        
At Risk                   42.5                 43.9           24.9                          63.3                                      4.7
Error                    100.0                 87.5            0.0                          17.0                                     66.0
Performing                90.5                 73.4            3.7                          28.9                                     43.1
Struggling                75.7                 64.3           11.1                          49.5                                     12.6
Thriving                  96.6                 86.6            1.0                          11.3                 

### 6b. Spot Check -- Find Obvious Misclassifications

In [18]:
check_cols = ['School Name', 'AI_Classification', 'AI_Insight',
              '% Graduated', '% Attending College', '% Dropped Out']
check_cols = [c for c in check_cols if c in hs.columns]

false_thriving = hs[(hs['AI_Classification'] == 'Thriving') & (hs['% Graduated'] < 80)][check_cols]
print(f'Schools labeled Thriving but graduation below 80%: {len(false_thriving)}')
print(false_thriving.to_string(index=False) if len(false_thriving) > 0 else 'None found -- good!')
print()

false_at_risk = hs[(hs['AI_Classification'] == 'At Risk') & (hs['% Graduated'] > 90)][check_cols]
print(f'Schools labeled At Risk but graduation above 90%: {len(false_at_risk)}')
print(false_at_risk.to_string(index=False) if len(false_at_risk) > 0 else 'None found -- good!')

Schools labeled Thriving but graduation below 80%: 0
None found -- good!

Schools labeled At Risk but graduation above 90%: 0
None found -- good!


### 6c. Consistency Check -- Does the AI Give the Same Answer Twice?

In [19]:
sample_schools = hs.sample(5, random_state=42)
consistency_results = []

print('Running consistency check -- classifying 5 schools twice each...')
print()

for _, row in sample_schools.iterrows():
    result_1 = classify_school(row, HS_PROMPT)
    time.sleep(0.5)
    result_2 = classify_school(row, HS_PROMPT)
    match = result_1['classification'] == result_2['classification']
    consistency_results.append({
        'School': row['School Name'],
        'Run 1': result_1['classification'],
        'Run 2': result_2['classification'],
        'Consistent': 'Yes' if match else 'No'
    })

consistency_df = pd.DataFrame(consistency_results)
print(consistency_df.to_string(index=False))

consistent_count = sum(1 for r in consistency_results if r['Consistent'] == 'Yes')
print(f'\nConsistency rate: {consistent_count}/5 ({consistent_count*20}%)')
print('90-100% -- prompt is reliable | 60-80% -- needs refinement | Below 60% -- do not use')

Running consistency check -- classifying 5 schools twice each...

                                    School      Run 1      Run 2 Consistent
                               Palmer High Performing Performing        Yes
Francis W. Parker Charter Essential School   Thriving   Thriving        Yes
         Four Rivers Charter Public School Performing Performing        Yes
                      West Roxbury Academy Struggling Struggling        Yes
     Global Learning Charter Public School Performing Performing        Yes

Consistency rate: 5/5 (100%)
90-100% -- prompt is reliable | 60-80% -- needs refinement | Below 60% -- do not use


---
# PART 7: Excel IF vs AI Classification Comparison

This section compares the AI classification to a rule-based Excel IF approach.  
The disagreements reveal where AI adds value over simple threshold rules.

In [29]:
def excel_classify(row):
    grad    = row.get('% Graduated', 0) or 0
    college = row.get('% Attending College', 0) or 0
    dropout = row.get('% Dropped Out', 0) or 0
    if grad >= 90 and college >= 70 and dropout <= 5:
        return 'Thriving'
    elif grad >= 75 and college >= 55 and dropout <= 10:
        return 'Performing'
    elif grad >= 60 and college >= 40 and dropout <= 20:
        return 'Struggling'
    else:
        return 'At Risk'

hs['IF_Classification'] = hs.apply(excel_classify, axis=1)

print('Excel IF Classification breakdown:')
print(hs['IF_Classification'].value_counts())
print()
print('AI Classification breakdown:')
print(hs['AI_Classification'].value_counts())

agreement = (hs['IF_Classification'] == hs['AI_Classification']).sum()
total = len(hs)
print(f'\nAgreement rate: {agreement}/{total} ({round(agreement/total*100, 1)}%)')

print('\n' + '='*80)
print('Schools where AI rated LOWER than Excel (AI caught problems Excel missed):')
ai_lower = hs[
    (hs['IF_Classification'].isin(['Thriving', 'Performing'])) &
    (hs['AI_Classification'].isin(['Struggling', 'At Risk'])) &
    (hs['AI_Classification'] != 'Error')
][['School Name', 'IF_Classification', 'AI_Classification',
   '% Graduated', '% Attending College', 'AI_Insight']]
print(ai_lower.to_string(index=False))

Excel IF Classification breakdown:
IF_Classification
Thriving      172
Performing     98
Struggling     41
At Risk        32
Name: count, dtype: int64

AI Classification breakdown:
AI_Classification
Performing    164
Thriving      115
Struggling     36
At Risk        27
Error           1
Name: count, dtype: int64

Agreement rate: 234/343 (68.2%)

Schools where AI rated LOWER than Excel (AI caught problems Excel missed):
                                    School Name IF_Classification AI_Classification  % Graduated  % Attending College                                                                                                                       AI_Insight
                                Lyon Upper 9-12        Performing        Struggling         88.2                 75.0                                                  The school faces challenges with low SAT scores despite a good graduation rate.
             Chicopee Comprehensive High School        Performing        Strugglin

---
# PART 8: AI Classification vs Official State Accountability Levels

The Massachusetts DOE classifies schools using an official Level 1-5 system.  
Comparing our AI classification to the state system reveals something unexpected.

**State Level definitions:**
```
Level 1 -- Meeting gap narrowing goals           521 schools
Level 2 -- Not meeting gap narrowing goals       787 schools
Level 3 -- Lowest performing 20% of subgroups   261 schools
Level 4 -- Underperforming                       31 schools
Level 5 -- Chronically underperforming            4 schools
Insufficient data                               232 schools
```

**Key difference:**
- Our AI measures **absolute outcomes** -- graduation rate, college attendance, SAT scores
- The state measures **equity** -- whether achievement gaps between demographic groups are closing

In [28]:
hs['State_Level'] = hs['Accountability and Assistance Level']

comparison = hs.groupby(['AI_Classification', 'State_Level']).size().unstack(fill_value=0)
print('AI Classification vs Official State Level (High Schools):')
print('='*70)
print(comparison.to_string())
print()

strong_agreement = hs[
    ((hs['AI_Classification'].isin(['Thriving', 'Performing'])) & (hs['State_Level'] == 'Level 1')) |
    ((hs['AI_Classification'].isin(['Struggling', 'At Risk'])) & (hs['State_Level'].isin(['Level 3', 'Level 4', 'Level 5'])))
]
print(f'Schools where AI and State broadly agree: {len(strong_agreement)}/{len(hs)}')
print(f'Agreement rate: {round(len(strong_agreement)/len(hs)*100, 1)}%')
print()
print('Low agreement is expected -- AI measures outcomes, state measures equity.')

AI Classification vs Official State Level (High Schools):
State_Level        Insufficient data  Level 1  Level 2  Level 3  Level 4
AI_Classification                                                       
At Risk                            9        0        0        9        9
Error                              0        0        1        0        0
Performing                         0       35      106       22        0
Struggling                         2        2        2       30        0
Thriving                           1       74       40        0        0

Schools where AI and State broadly agree: 157/343
Agreement rate: 45.8%

Low agreement is expected -- AI measures outcomes, state measures equity.


In [27]:
# Headline finding: schools that look great on outcomes but are failing on equity
thriving_but_level2 = hs[
    (hs['AI_Classification'] == 'Thriving') &
    (hs['State_Level'] == 'Level 2')
][['School Name', 'Town', '% Graduated', '% Attending College',
   '% Economically Disadvantaged', 'State_Level', 'AI_Insight']]

print(f'HEADLINE FINDING: {len(thriving_but_level2)} schools are Thriving on outcomes but Level 2 on equity')
print('These schools have strong graduation and college rates but are not closing achievement gaps.')
print('='*80)
print(thriving_but_level2.to_string(index=False))

HEADLINE FINDING: 40 schools are Thriving on outcomes but Level 2 on equity
These schools have strong graduation and college rates but are not closing achievement gaps.
                                         School Name               Town  % Graduated  % Attending College  % Economically Disadvantaged State_Level                                                                                                        AI_Insight
                                        Andover High            Andover         95.7                 89.3                           6.3     Level 2                         Andover High boasts an impressive graduation rate of 95.7% and strong college attendance.
                                        Boston Latin             Boston         98.1                 94.1                          15.7     Level 2         Boston Latin boasts exceptional graduation and college attendance rates alongside high SAT and AP scores.
                                Boston Latin 

In [30]:
at_risk_and_level3 = hs[
    (hs['AI_Classification'] == 'At Risk') &
    (hs['State_Level'].isin(['Level 3', 'Level 4', 'Level 5']))
]

print('KEY FINDING: Outcome Performance vs Equity Performance')
print('='*60)
print(f'Thriving on outcomes BUT Level 2 on equity : {len(thriving_but_level2)} schools')
print(f'At Risk on outcomes AND Level 3/4 on equity: {len(at_risk_and_level3)} schools')
print()
print('A school can have excellent graduation rates while still failing')
print('its most vulnerable students. This is invisible if you only look')
print('at one classification system -- it only emerges when you compare both.')

KEY FINDING: Outcome Performance vs Equity Performance
Thriving on outcomes BUT Level 2 on equity : 40 schools
At Risk on outcomes AND Level 3/4 on equity: 18 schools

A school can have excellent graduation rates while still failing
its most vulnerable students. This is invisible if you only look
at one classification system -- it only emerges when you compare both.


---
# PART 9: Analyst Commentary

### What the AI Got Right
- **Perfect tier progression:** Thriving schools averaged 97.2% graduation vs 41.2% for At Risk -- the classifications are statistically sound
- **Zero misclassifications** in spot check -- no school falsely labeled Thriving with low graduation rates or At Risk with high ones
- **100% consistency rate** -- the same school classified twice always got the same result

### Where the AI Struggled
- Some schools returned errors due to missing or unusual data formatting -- excluded from final analysis
- The AI is a black box compared to Excel IF functions -- I cannot see exactly why it made each decision, which is why validation was essential

### Excel IF vs AI -- Which Do I Trust More?

When I compared rule-based Excel classification to AI classification for high schools they only agreed around 53% of the time. Investigating the disagreements revealed something important:

- Excel classified schools purely on graduation rate thresholds
- The AI caught deeper patterns -- like schools with acceptable graduation rates but dangerously low SAT scores and AP performance
- Example: Claremont Academy passed Excel thresholds but AI flagged it as Struggling due to low SAT and AP performance

I trust the AI classification more because it evaluated each school across 15+ metrics simultaneously. The Excel IF approach only used 2-3 metrics because adding more makes the formula exponentially complex.

That said, AI is not a replacement for human judgment. Excel IF is fully transparent -- you can see exactly why every school got its label. With AI you have to validate the output rather than trust the logic directly. That is exactly why I ran consistency checks, spot checks and tier validation before using these classifications in Tableau.

### How I Would Improve the Prompt
- Add explicit numeric thresholds as guidelines to reduce ambiguity for edge cases
- Ask the model to explain its reasoning so misclassifications are easier to investigate
- Handle missing data more gracefully to eliminate error cases

### AI Classification vs Official State Accountability Levels

When I compared my AI outcome-based classification to the Massachusetts state official accountability levels an unexpected finding emerged.

The overall agreement rate was 46.1% -- which sounds low but makes sense because the two systems are measuring different things. My AI measures absolute student outcomes. The state measures equity -- whether achievement gaps are closing between demographic groups.

**The most important finding in this entire project:**

46 schools were classified as Thriving by the AI but rated Level 2 by the state. Schools like Boston Latin (98.1% graduation) and Andover High (95.7% graduation) look exceptional on paper -- but the state flagged them for not closing achievement gaps. These are schools that are excellent for advantaged students but are not lifting disadvantaged students at the same rate.

16 schools are At Risk on outcomes AND Level 3 or 4 on equity -- failing on both measures simultaneously and in most need of intervention.

This distinction is invisible if you only look at one classification system. It only emerges when you compare both.

### My Overall Confidence in the AI Output
**High.** The classifications are statistically validated, consistent, and free of obvious misclassifications. The comparison with state accountability levels revealed a genuinely important insight about the difference between outcome performance and equity performance -- a finding that would have been invisible without building our own classification system.

---
# PART 10: Export Enriched Datasets for Tableau

In [31]:
# Export 1: High schools with all original columns plus AI and IF classification
hs.to_csv('MA_HighSchools_AI_Enriched.csv', index=False)
print(f'High schools exported: {len(hs)} rows')

# Export 2: K-8 schools with all original columns plus AI classification
k8.to_csv('MA_K8Schools_AI_Enriched.csv', index=False)
print(f'K-8 schools exported: {len(k8)} rows')

# Export 3: Middle schools with all original columns plus AI classification
middle.to_csv('MA_MiddleSchools_AI_Enriched.csv', index=False)
print(f'Middle schools exported: {len(middle)} rows')

# Export 4: All school types combined for the state overview dashboard
all_schools.to_csv('MA_AllSchools_AI_Enriched.csv', index=False)
print(f'All schools combined exported: {len(all_schools)} rows')
print()
print('Files ready for Tableau:')
print('  MA_HighSchools_AI_Enriched.csv   -- graduation and college analysis')
print('  MA_K8Schools_AI_Enriched.csv     -- 3rd and 4th grade MCAS analysis')
print('  MA_MiddleSchools_AI_Enriched.csv -- 6th, 7th and 8th grade MCAS analysis')
print('  MA_AllSchools_AI_Enriched.csv    -- state overview and classification map')
print()
print('Key columns for Tableau:')
print('  AI_Classification  -- Color on Marks card')
print('  AI_Insight         -- Tooltip')
print('  AI_Key_Strength    -- Dashboard callouts')
print('  AI_Key_Concern     -- Dashboard callouts')
print('  School_Level       -- Filter between High School, K-8, Middle School')
print('  IF_Classification  -- Comparison view (high schools file only)')




High schools exported: 343 rows
K-8 schools exported: 254 rows
Middle schools exported: 81 rows
All schools combined exported: 678 rows

Files ready for Tableau:
  MA_HighSchools_AI_Enriched.csv   -- graduation and college analysis
  MA_K8Schools_AI_Enriched.csv     -- 3rd and 4th grade MCAS analysis
  MA_MiddleSchools_AI_Enriched.csv -- 6th, 7th and 8th grade MCAS analysis
  MA_AllSchools_AI_Enriched.csv    -- state overview and classification map

Key columns for Tableau:
  AI_Classification  -- Color on Marks card
  AI_Insight         -- Tooltip
  AI_Key_Strength    -- Dashboard callouts
  AI_Key_Concern     -- Dashboard callouts
  School_Level       -- Filter between High School, K-8, Middle School
  IF_Classification  -- Comparison view (high schools file only)


---
## Summary

### What This Notebook Demonstrates

| Skill | How It Shows Up |
|---|---|
| Data preparation | Filtered 1,861 schools into three subgroups using metric-based logic |
| Analytical thinking | Built separate classification frameworks for each school type |
| Prompt engineering | Designed structured prompts returning consistent JSON across 678 schools |
| API integration | Called OpenAI API programmatically via Python at scale |
| Critical thinking | Validated, spot-checked and stress-tested AI output before trusting it |
| Comparative analysis | Compared AI vs Excel IF and AI vs state official classification |
| Insight generation | Discovered the outcomes vs equity gap affecting 46 Massachusetts high schools |
| Communication | Exported enriched data into Tableau for storytelling |

### Coverage
```
Total schools in dataset:    1,861
High schools classified:       343  (graduation, college, SAT, AP metrics)
K-8 schools classified:        254  (3rd and 4th grade MCAS metrics)
Middle schools classified:      81  (6th, 7th and 8th grade MCAS metrics)
Total classified:              678  (36% of all Massachusetts schools)
Excluded:                    1,183  (early childhood, incomplete data, alternative schools)
```

### Key Findings
1. **Outcomes vs Equity gap:** 46 high schools are Thriving on graduation and college attendance but Level 2 on the state equity metric -- excellent for advantaged students but not closing achievement gaps
2. **AI vs Excel:** AI and rule-based classification agreed only ~67.3% of the time -- AI caught nuances like low SAT and AP performance that simple graduation thresholds missed
3. **Worst performing schools:** 16 schools are simultaneously At Risk on outcomes AND Level 3/4 on equity -- failing on both measures and most in need of intervention

### Limitations

**1. Single year snapshot -- no trend analysis**
The entire analysis is based on 2017 data only. We cannot tell if a school is improving or declining over time. A school classified as At Risk might be on an upward trajectory while a Thriving school might be declining. Multi-year data would transform this from a static snapshot into a dynamic story about which schools are getting better and which are getting worse.

**2. Coverage is 36% of total schools**
1,183 schools were excluded because they lacked sufficient relevant data for their school type. Early childhood centers have no academic outcome metrics at all. Classifying them would produce misleading results based on insufficient data. Coverage is intentionally conservative rather than artificially complete.

**3. Column selection based on judgment not statistics**
The 19 columns for high schools and equivalent columns for K-8 and middle schools were selected based on domain knowledge and reasoning about what matters in education. We did not run a formal feature selection analysis like a correlation test to statistically validate which columns most strongly predict school quality. That would be the more rigorous approach and is a candidate for future improvement.

**4. AI is a black box**
We cannot see exactly why the AI made each individual classification decision. Two schools with very similar metrics could theoretically receive different classifications. This is exactly why the validation section exists -- tier progression check, spot check and consistency check all confirmed the output was reliable before using it in Tableau.

**5. Multi-level schools classified into one bucket**
K-12, 6-12 and 7-12 schools are classified purely on their high school metrics -- graduation rate and college attendance. Their elementary and middle school performance is completely ignored. Ideally these schools would receive three separate classifications but the dataset has one row per school not one row per grade level.

### Files
| File | Description |
|---|---|
| `MA_Schools_AI_Analysis_v4.ipynb` | Main analysis notebook |
| `MA_HighSchools_AI_Enriched.csv` | 343 high schools with all columns plus AI and IF classification |
| `MA_K8Schools_AI_Enriched.csv` | 254 K-8 schools with all columns plus AI classification |
| `MA_MiddleSchools_AI_Enriched.csv` | 81 middle schools with all columns plus AI classification |
| `MA_AllSchools_AI_Enriched.csv` | 678 schools combined with 16 common columns for Tableau overview |
| `MA_Schools_data_set.xlsx` | Original dataset from the Massachusetts DOE |

---

---
*Chalkboards and Dashboards -- MA Schools Analysis 2017*